Basic settings

In [ ]:
from utgc.runner import EnsembleRunner
runner = EnsembleRunner.from_config('output/SA/testrun/config.yaml')

Election settings

In [ ]:
runner = (runner
    .add_election_updater(
        name="2016PRE",
        parties_to_columns={"D":"G16PREDCLI","R":"G16PRERTRU","-":"G16PRE-TOT"}
    )
    .add_election_updater(
        name="2016GOV",
        parties_to_columns={"D":"G16GOVDWEI","R":"G16GOVRHER","-":"G16GOV-TOT"}
    )
    .add_election_updater(
        name="2016ATG",
        parties_to_columns={"D":"G16ATGDHAR","R":"G16ATGRREY","-":"G16ATG-TOT"}
    )
    .add_election_updater(
        name="2016AUD",
        parties_to_columns={"D":"G16AUDDMIT","R":"G16AUDRDOU","-":"G16AUD-TOT"}
    )
    .add_election_updater(
        name="2016TRE",
        parties_to_columns={"D":"G16TREDHAN","R":"G16TRERDAM","-":"G16TRE-TOT"}
    )
    .add_election_updater(
        name="2020PRE",
        parties_to_columns={"D":"G20PREDBID","R":"G20PRERTRU","-":"G20PRE-TOT"}
    )
    .add_election_updater(
        name="2020GOV",
        parties_to_columns={"D":"G20GOVDPET","R":"G20GOVRCOX","-":"G20GOV-TOT"}
    )
    .add_election_updater(
        name="2020ATG",
        parties_to_columns={"D":"G20ATGDSKO","R":"G20ATGRREY","-":"G20ATG-TOT"}
    )
    .add_election_updater(
        name="2024PRE",
        parties_to_columns={"D":"G24PREDHAR","R":"G24PRERTRU","-":"G24PRE-TOT"}
    )
    .add_election_updater(
        name="2024GOV",
        parties_to_columns={"D":"G24GOVDCUM","R1":"G24GOVRHEN","R2":"G24GOVNCLA","-":"G24GOV-TOT"}
    )
    .add_election_updater(
        name="2024ATG",
        parties_to_columns={"D":"G24ATGDBAU","R":"G24ATGRBRO","-":"G24ATG-TOT"}
    )
    .add_election_updater(
        name="2024AUD",
        parties_to_columns={"D":"G24AUDDVOU","R":"G24AUDRCAN","-":"G24AUD-TOT"}
    )
    .add_election_updater(
        name="2024TRE",
        parties_to_columns={"D":"G24TREDHAN","R":"G24TREROAK","-":"G24TRE-TOT"}  
    )
    .add_election_aggregator(
        "sb1011_data",
        ["2016PRE", "2016GOV", "2016ATG", "2016AUD", "2016TRE", "2020PRE", "2020GOV", "2020ATG", "2024PRE", "2024GOV", "2024ATG", "2024AUD", "2024TRE"],
        parties=["D", "R", "-"]
    )
    .add_election_metric_updaters(
        "sb1011_data",
        ["partisan_bias_utah", "partisan_bias", "mean_median", "efficiency_gap", "stdev_partisan_share", "majority_partisan_shares", "majority_seats"]
    )
)

In [ ]:
# Set up visualization of sample maps
import os
import geopandas as gpd
import utgc.plotting as gcplot
munis = gpd.read_file("data/geography/UtahMunicipalBoundaries/Municipalities.shp").to_crs(runner.geodata.crs)
counties = gpd.read_file("data/geography/UtahCountyBoundaries/ut_cnty_2020_bound.shp").to_crs(runner.geodata.crs)

runner = runner.add_runtime_callback(
    name="render_maps",
    frequency=10,
    action=lambda partition, step, output_dir: gcplot.visualize_partition(
        partition,
        step,
        os.path.join(output_dir, "maps"),
        counties=counties,
        municipalities=munis,
        split_munis_count=partition["split_muni"],
        split_counties_count=partition["split_county"],
    )
)

In [ ]:
runner = runner.precondition(steps=50)
runner.run(
    name="ensemble",
    output_dir="output/SA/",
    num_steps=5000,
)

In [ ]:
from utgc.results import ResultSet
import matplotlib.pyplot as plt
results = ResultSet(output_file="output/SA/ensemble/output.jsonl")

results.plot_metric_violin("majority_partisan_shares")